# 10. Expanded text models

Owner: rdevasia25. Issue #10. Depends on #5 and #9.

## Goal

Compare the fixed ten-candidate text grid using five training folds. Select a small margin ensemble by OOF AUROC, then confirm on the fixed validation rows.

## Setup

Longer word patterns and case-sensitive fragments may add signal. Keep weaker candidates, including the nine-style arm that failed its earlier adoption rule.

In [ ]:
import os, re, json, time, warnings, gc
from pathlib import Path
import cloudpickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from IPython import get_ipython
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
EXPANDED_HELPERS_ONLY = os.environ.pop("EXPANDED_TEXT_HELPERS_ONLY", "0") == "1"
get_ipython().run_line_magic("run", "./02_split_metrics.ipynb")
EXPANDED_GRID = [
    {"name": "word12_lr4", "view": "word12", "learner": "lr", "C": 4.},
    {"name": "word12_lr16", "view": "word12", "learner": "lr", "C": 16.},
    {"name": "word13_lr16", "view": "word13", "learner": "lr", "C": 16.},
    {"name": "word12_svm1", "view": "word12", "learner": "svm", "C": 1.},
    {"name": "charwb_lr16", "view": "charwb", "learner": "lr", "C": 16.},
    {"name": "casechar_lr16", "view": "casechar", "learner": "lr", "C": 16.},
    {"name": "casechar_svm1", "view": "casechar", "learner": "svm", "C": 1.},
    {"name": "word13_casechar_lr16", "view": "word_char", "learner": "lr", "C": 16.},
    {"name": "nbword13_lr4", "view": "nbword13", "learner": "lr", "C": 4.},
    {"name": "word12_style_lr4", "view": "word_style", "learner": "lr", "C": 4.},
]
EXPANDED_VIEWS = {
    "word12": {"analyzer": "word", "ngram_range": [1, 2], "min_df": 2, "max_features": 50000, "lowercase": True},
    "word13": {"analyzer": "word", "ngram_range": [1, 3], "min_df": 2, "max_features": None, "lowercase": True},
    "charwb": {"analyzer": "char_wb", "ngram_range": [3, 5], "min_df": 2, "max_features": 100000, "lowercase": True},
    "casechar": {"analyzer": "char", "ngram_range": [2, 5], "min_df": 3, "max_features": 300000, "lowercase": False},
}
EXPANDED_LEARNERS = {"lr": {"solver": "liblinear", "max_iter": 5000, "random_state": 42, "tol": 1e-4},
                     "svm": {"dual": "auto", "max_iter": 20000, "random_state": 42, "tol": 1e-4}}


## Checks

Verify raw bytes and fixed split with notebook 02. Five shuffled stratified folds use seed 42 within the 8,000 training rows. Feature scaling and NB class ratios fit only on each fit partition. The issue explicitly uses eligible unlabeled text as a label-free margin reference; no test-batch normalization.

In [ ]:
if not EXPANDED_HELPERS_ONLY:
    NAME = "10_expanded_text"
    COMMAND = "uv run --locked jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=3600 notebooks/10_expanded_text.ipynb"
    START = time.perf_counter()
    parts = load_partitions()
    train, validation, reference = parts["train"], parts["validation"], parts["unlabeled"]
    require(len(reference) > 0, "No eligible margin reference")
    folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train.text, train.label))
    CONFIG = {"grid": EXPANDED_GRID, "views": EXPANDED_VIEWS, "learners": EXPANDED_LEARNERS,
        "tfidf": {"sublinear_tf": True, "norm": "l2", "dtype": "float32", "stop_words": None},
        "nb": {"binary": True, "min_df": 2, "ngram_range": [1, 3], "max_features": None,
               "alpha": 1., "ratio": "log normalized class token frequency ratio; training only"},
        "style": "exact nine features from notebook 09; StandardScaler fitted inside each training pipeline",
        "margin_reference": {"rows": len(reference), "ordered_id_sha256": value_hash(reference.id.tolist()),
                             "definition": "eligible unlabeled only; population mean/std; zero-variance reference fails"},
        "cv": {"n_splits": 5, "shuffle": True, "seed": 42, "selection_metric": "pooled OOF AUROC"},
        "ensemble": {"min_gain": .0005, "max_members": 5, "weights": "equal", "repeat_members": False,
                     "tie": "grid order; prefer fewer members within .0005 of best path score"},
        "threads": 1, "score_mapping": "sigmoid(mean standardized margins); not claimed calibrated",
        "fold_id_sha256": [{"fit": value_hash(train.id.iloc[a].tolist()), "score": value_hash(train.id.iloc[b].tolist())} for a,b in folds],
        "style_notebook_source_sha256": value_hash(["".join(c["source"]) for c in json.loads((ROOT / "notebooks/09_linear_style.ipynb").read_text())["cells"]]),
        "notebook_source_sha256": value_hash(["".join(c["source"]) for c in json.loads((ROOT / "notebooks/10_expanded_text.ipynb").read_text())["cells"]]),
    }
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    private = ROOT / "artifacts" / NAME / RUN_ID
    private.mkdir(parents=True, exist_ok=False)


## Method

TF-IDF views keep sublinear TF, L2 normalization, and float32. NB uses binary word counts multiplied by a supervised log frequency ratio with alpha 1. The nine-style block matches #9 and scales inside each fit. Case character features preserve punctuation, whitespace, and case.

Each candidate gets fresh vocabulary and model fits in each fold. Standardize its margin using the same eligible unlabeled reference, save those mean/std values, and average standardized margins. The sigmoid supplies bounded scores, not a claim of calibrated probabilities.

In [ ]:
class NBLogRatio(TransformerMixin, BaseEstimator):
    def __init__(self, alpha=1.):
        self.alpha = alpha
    def fit(self, X, y):
        y = np.asarray(y)
        require(set(y) == {0, 1}, "NB ratio needs both classes")
        positive = np.asarray(X[y == 1].sum(axis=0)).ravel().astype(np.float64) + self.alpha
        negative = np.asarray(X[y == 0].sum(axis=0)).ravel().astype(np.float64) + self.alpha
        self.ratio_ = np.log((positive / positive.sum()) / (negative / negative.sum())).astype(np.float32)
        return self
    def transform(self, X):
        return X.multiply(self.ratio_).tocsr()

NINE_SENTENCE_END = re.compile(r"[.!?]+")
NINE_LIST_LINE = re.compile(r"^\s*(?:[-*\u2022]|\d+[.)])\s", re.MULTILINE)
NINE_TOKEN = re.compile(r"\w+")
def nine_style_features(texts):
    rows = []
    for text in texts:
        text = str(text)
        words, chars = len(text.split()), max(1, len(text))
        letters = sum(c.isalpha() for c in text)
        tokens = NINE_TOKEN.findall(text.lower())
        rows.append([np.log1p(words), text.count("\n") / max(1, words),
            sum(not c.isalnum() and not c.isspace() for c in text) / chars,
            sum(c.isupper() for c in text) / max(1, letters), sum(c.isdigit() for c in text) / chars,
            np.log1p(words / max(1, len(NINE_SENTENCE_END.findall(text)))),
            len(set(tokens)) / max(1, len(tokens)), sum(ord(c) > 127 for c in text) / chars,
            len(NINE_LIST_LINE.findall(text)) / max(1, text.count("\n") + 1)])
    return np.asarray(rows, dtype=np.float32)

def expanded_vectorizer(view):
    settings = dict(EXPANDED_VIEWS[view])
    settings["ngram_range"] = tuple(settings["ngram_range"])
    return TfidfVectorizer(**settings, sublinear_tf=True, norm="l2", dtype=np.float32)

def make_expanded_model(candidate):
    view = candidate["view"]
    if view == "word_char":
        features = FeatureUnion([("word", expanded_vectorizer("word13")), ("char", expanded_vectorizer("casechar"))], n_jobs=1)
    elif view == "word_style":
        style = Pipeline([("measure", FunctionTransformer(nine_style_features)), ("scale", StandardScaler())])
        features = FeatureUnion([("word", expanded_vectorizer("word12")), ("style", style)], n_jobs=1)
    elif view == "nbword13":
        features = Pipeline([("counts", CountVectorizer(ngram_range=(1, 3), min_df=2, binary=True, dtype=np.float32)),
                             ("ratio", NBLogRatio(alpha=1.))])
    else:
        features = expanded_vectorizer(view)
    learner = candidate["learner"]
    classifier = (LogisticRegression if learner == "lr" else LinearSVC)(C=candidate["C"], **EXPANDED_LEARNERS[learner])
    return Pipeline([("features", features), ("classifier", classifier)])

def margin_reference(model, texts):
    require(list(model.classes_) == [0, 1], "Wrong margin orientation")
    margin = np.asarray(model.decision_function(texts), dtype=float)
    require(np.isfinite(margin).all(), "Nonfinite margin reference")
    mean, std = float(margin.mean()), float(margin.std())
    require(std > 1e-12, "Degenerate margin reference")
    return {"mean": mean, "std": std}

def standardized_margin(model, texts, stats):
    return (np.asarray(model.decision_function(texts), dtype=float) - stats["mean"]) / stats["std"]

class MarginEnsemble:
    classes_ = np.array([0, 1])
    def __init__(self, models, statistics):
        self.models, self.statistics = models, statistics
    def decision_function(self, texts):
        return np.mean([standardized_margin(m, texts, s) for m,s in zip(self.models, self.statistics)], axis=0)
    def predict_proba(self, texts):
        score = expit(self.decision_function(texts))
        return np.column_stack([1 - score, score])

def greedy_margin_selection(oof, labels, names, min_gain=.0005, max_members=5):
    individual = [float(roc_auc_score(labels, oof[name])) for name in names]
    selected = [names[int(np.argmax(individual))]]
    path = [{"members": selected.copy(), "auroc": max(individual)}]
    trials = []
    while len(selected) < min(max_members, len(names)):
        choices = []
        for name in names:
            if name in selected:
                continue
            members = selected + [name]
            score = float(roc_auc_score(labels, np.mean([oof[k] for k in members], axis=0)))
            choices.append((name, score))
            trials.append({"members": members, "auroc": score, "gain": score - path[-1]["auroc"]})
        name, score = max(choices, key=lambda item: item[1])
        if score - path[-1]["auroc"] < min_gain:
            break
        selected.append(name)
        path.append({"members": selected.copy(), "auroc": score})
    best = max(item["auroc"] for item in path)
    chosen = next(item for item in path if item["auroc"] >= best - min_gain - 1e-12)
    return {"selected": chosen["members"], "selected_oof_auroc": chosen["auroc"], "path": path, "trials": trials}


## Run

Commit the grid before this cell runs. Save the chosen ensemble before any validation scoring. Keep every candidate, all OOF rows privately, all reference statistics, and both CV and full-fit runtimes.

In [ ]:
if not EXPANDED_HELPERS_ONLY:
    # Synthetic checks target learned ratio direction, empty text, and greedy stopping.
    fixture = np.array([[2., 0.], [1., 0.], [0., 1.], [0., 2.]], dtype=np.float32)
    from scipy.sparse import csr_matrix
    ratio = NBLogRatio().fit(csr_matrix(fixture), [0, 0, 1, 1]).ratio_
    require(ratio[0] < 0 < ratio[1], "NB ratio orientation failed")
    require(np.array_equal(nine_style_features([""]), np.zeros((1,9), dtype=np.float32)), "Empty style features failed")
    toy_selection = greedy_margin_selection({"a": np.array([0., 1., 2., 3.]), "b": np.array([0., 1., 2., 3.])}, [0,0,1,1], ["a","b"])
    require(toy_selection["selected"] == ["a"], "Greedy tie or gain rule failed")
    plan_path = ROOT / "results" / NAME / f"{RUN_ID}-plan.json"
    save_new_json(plan_path, {"config": CONFIG, "saved_before_fit": True, "config_sha256": value_hash(CONFIG), "split_sha256": parts["split"]["split_sha256"]})
    artifacts = [plan_path]
    cv_rows, oof, candidate_rows = [], {}, []
    try:
        search_start = time.perf_counter()
        with threadpool_limits(limits=1), warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            for candidate in EXPANDED_GRID:
                name = candidate["name"]
                oof[name] = np.full(len(train), np.nan)
                candidate_start = time.perf_counter()
                for fold, (a,b) in enumerate(folds):
                    start = time.perf_counter()
                    model = make_expanded_model(candidate).fit(train.text.iloc[a], train.label.iloc[a])
                    stats = margin_reference(model, reference.text)
                    score = standardized_margin(model, train.text.iloc[b], stats)
                    require(np.isfinite(score).all(), "Invalid OOF score")
                    oof[name][b] = score
                    cv_rows.append({"candidate": name, "fold": fold, "auroc": float(roc_auc_score(train.label.iloc[b], score)),
                                    "seconds": time.perf_counter() - start, "reference": stats})
                    del model; gc.collect()
                values = [row["auroc"] for row in cv_rows if row["candidate"] == name]
                candidate_rows.append({"candidate": name, "pooled_oof_auroc": float(roc_auc_score(train.label, oof[name])),
                    "fold_mean": float(np.mean(values)), "fold_std": float(np.std(values)), "cv_seconds": time.perf_counter() - candidate_start})
                print(f"Completed training CV: {name}")
        search_seconds = time.perf_counter() - search_start
        oof_path = private / "oof_margins.csv"
        pd.DataFrame({"id": train.id, **oof}).to_csv(oof_path, index=False); artifacts.append(oof_path)
        selection = greedy_margin_selection(oof, train.label, [c["name"] for c in EXPANDED_GRID])
        selection_path = ROOT / "results" / NAME / f"{RUN_ID}-selection.json"
        save_new_json(selection_path, {**selection, "candidate_results": candidate_rows, "validation_inspected": False,
            "config_sha256": value_hash(CONFIG), "utc": datetime.now(timezone.utc).isoformat()}); artifacts.append(selection_path)
        selected_models, selected_stats, saved_candidates, final_rows = [], [], {}, []
        with threadpool_limits(limits=1), warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            for candidate in EXPANDED_GRID:
                name = candidate["name"]
                start = time.perf_counter()
                model = make_expanded_model(candidate).fit(train.text, train.label)
                fit_seconds = time.perf_counter() - start
                stats = margin_reference(model, reference.text)
                evaluation_start = time.perf_counter()
                train_score = standardized_margin(model, train.text, stats)
                valid_score = standardized_margin(model, validation.text, stats)
                final_rows.append({"candidate": name, "train_auroc": float(roc_auc_score(train.label, train_score)),
                    "validation_auroc": float(roc_auc_score(validation.label, valid_score)), "fit_seconds": fit_seconds,
                    "evaluation_seconds": time.perf_counter() - evaluation_start, "reference": stats})
                path = private / f"{name}.pkl"
                with path.open("wb") as handle: cloudpickle.dump(MarginEnsemble([model], [stats]), handle)
                artifacts.append(path)
                saved_candidates[name] = {"model": str(path.relative_to(ROOT)), "config": candidate, "reference": stats}
                for partition, frame, scores in [("train",train,train_score),("validation",validation,valid_score)]:
                    path = private / f"{name}_{partition}.csv"
                    pd.DataFrame({"id": frame.id, "margin": scores, "p_ai": expit(scores)}).to_csv(path,index=False); artifacts.append(path)
                if name in selection["selected"]:
                    selected_models.append(model); selected_stats.append(stats)
                else:
                    del model
                gc.collect()
        ensemble = MarginEnsemble(selected_models, selected_stats)
        train_pred = pd.DataFrame({"id":train.id, "p_ai":class1_probabilities(ensemble, train.text)})
        valid_pred = pd.DataFrame({"id":validation.id, "p_ai":class1_probabilities(ensemble, validation.text)})
        train_metrics, valid_metrics = evaluate_predictions(train, train_pred), evaluate_predictions(validation, valid_pred)
        model_path = private / "ensemble.pkl"
        with model_path.open("wb") as handle: cloudpickle.dump(ensemble, handle)
        with model_path.open("rb") as handle: restored = cloudpickle.load(handle)
        with threadpool_limits(limits=1):
            reload_difference = float(np.max(np.abs(class1_probabilities(restored, validation.text) - valid_pred.p_ai.to_numpy())))
        require(reload_difference <= 1e-12, "Reload changed ensemble")
        artifacts.append(model_path)
        for partition, frame in [("train",train_pred),("validation",valid_pred)]:
            path = private / f"ensemble_{partition}.csv"; frame.to_csv(path,index=False); artifacts.append(path)
    except Exception as exc:
        record_run(NAME,COMMAND,CONFIG,SEED,len(train),len(validation),{"validation_auroc":None,"completed_folds":cv_rows},
                   time.perf_counter()-START,status="failed",error_type=type(exc).__name__)
        raise RuntimeError("Expanded search failed; see safe record and private execution log.") from None


## Results

OOF selection is a training estimate. Validation has been reused across project comparisons, so it is not an untouched final test. Margin standardization fixes scale for averaging; it does not calibrate probabilities or establish robust behavior under changed formatting.

In [ ]:
if not EXPANDED_HELPERS_ONLY:
    display(pd.DataFrame(candidate_rows)); display(pd.DataFrame(final_rows))
    display(pd.DataFrame(selection["path"]))
    print("Selected:", selection["selected"])
    print(f"Ensemble train AUROC {train_metrics['auroc']:.6f}; validation {valid_metrics['auroc']:.6f}; search {search_seconds:.2f}s")
    fig, ax = plt.subplots(figsize=(10,4))
    x=np.arange(len(candidate_rows))
    ax.bar(x-.18,[r["pooled_oof_auroc"] for r in candidate_rows],width=.36,label="training OOF")
    ax.bar(x+.18,[r["validation_auroc"] for r in final_rows],width=.36,label="validation")
    ax.set(xticks=x,xticklabels=[r["candidate"] for r in candidate_rows],ylim=(0,1),ylabel="AUROC",title="Expanded text grid")
    ax.tick_params(axis="x",rotation=65); ax.legend(); fig.tight_layout()
    plot_path=private/"comparison.png";fig.savefig(plot_path,dpi=140);artifacts.append(plot_path);plt.show()


## Takeaway

- Tried the complete fixed grid and a guarded equal-margin ensemble.
- Tables retain all measured candidates, fold variation, cost, and the saved selection path.
- Next: #11 tests formatting rules. Keep reference means/stds fixed at inference and do not normalize test batches.

## Save

Private cloudpickle models contain their notebook-defined feature code. `artifacts/10_expanded_text/latest.json` exposes the selected ensemble, all candidate settings, reference statistics, and ID-aligned predictions. Set `EXPANDED_TEXT_HELPERS_ONLY=1` before running this notebook to load definitions without fitting or writing.

In [ ]:
if not EXPANDED_HELPERS_ONLY:
    CONFIG["extra_versions"]={k:importlib.metadata.version(k) for k in ["cloudpickle","matplotlib","threadpoolctl"]}
    metrics={"validation_auroc":valid_metrics["auroc"],"train_auroc":train_metrics["auroc"],
        "selected":selection["selected"],"selected_oof_auroc":selection["selected_oof_auroc"],
        "candidate_results":candidate_rows,"fold_results":cv_rows,"final_results":final_rows,
        "selection_path":selection["path"],"search_seconds":search_seconds,"max_reload_difference":reload_difference}
    record_path=record_run(NAME,COMMAND,CONFIG,SEED,len(train),len(validation),metrics,time.perf_counter()-START,artifacts=artifacts)
    pointer={"result":str(record_path.relative_to(ROOT)),"result_sha256":file_hash(record_path),
        "split_sha256":parts["split"]["split_sha256"],"model":str(model_path.relative_to(ROOT)),
        "selected":selection["selected"],"candidates":saved_candidates,
        "train_predictions":str((private/"ensemble_train.csv").relative_to(ROOT)),
        "validation_predictions":str((private/"ensemble_validation.csv").relative_to(ROOT)),
        "class1_interface":"class1_probabilities(cloudpickle.load(handle), texts)",
        "artifacts":{str(p.relative_to(ROOT)):file_hash(p) for p in artifacts}}
    target=ROOT/"artifacts"/NAME/"latest.json"; temporary=target.with_suffix(".tmp")
    temporary.write_text(json.dumps(pointer,sort_keys=True,indent=2)+"\n");temporary.replace(target)
    print("Saved expanded text comparison and private handoff.")
